# Revise existing indications from a two-label source diff

This notebook prototypes revision detection without generating latest-label indications. It compares the raw Indications and Usage sections from the label referenced by the current MOAlmanac document and the newest label, then asks one LLM call which existing indications were revised and how.

New and removed indication discovery is intentionally a separate workflow.

In [ ]:
import json
import os
from collections import Counter
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

from moalmanac_fda_curation.core.identify_revised_indications import (
    build_revision_identification_prompt,
    build_section_diff_hunks,
    identify_revised_indications,
    load_section_pair_from_cache,
)
from moalmanac_fda_curation.core.artifacts import document_label_url
from moalmanac_fda_curation.core.propose_revised_indication import (
    build_label_diff_revision_prompt,
    propose_revised_indication,
)
from moalmanac_fda_curation.core.identify_new_indications import (
    load_existing_indications,
)

## Configure the Opdivo replay

The baseline is the concrete label URL in the current Opdivo document artifact. The latest URL is selected deterministically from the newest event in the existing Opdivo changelog. Both raw Section 1 snapshots come from the existing cache created by the repository's label extraction code.

In [ ]:
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("No .env file found")
load_dotenv(env_path)
if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(f"ANTHROPIC_API_KEY is missing from {env_path}")

PROJECT_ROOT = Path(env_path).parent
WORKSPACE_ROOT = PROJECT_ROOT.parent
EXISTING_INDICATIONS_JSON = WORKSPACE_ROOT / "moalmanac-db/referenced/indications.json"
BASELINE_DOCUMENT_JSON = PROJECT_ROOT / "analyses/revisions/opdivo-current-record/document.json"
CHANGELOG_JSON = WORKSPACE_ROOT / "ai-assisted-gk-curation/analyses/fda-indications/extracted-indications/section1-changelogs/Opdivo-bla125554-section1-changelog.json"
SECTION_CACHE_JSON = WORKSPACE_ROOT / "ai-assisted-gk-curation/analyses/fda-indications/extracted-indications/Opdivo-bla125554-section1-cache.json"

In [ ]:
baseline_document = json.loads(BASELINE_DOCUMENT_JSON.read_text())
baseline_label_url = document_label_url(baseline_document)
changelog = json.loads(CHANGELOG_JSON.read_text())
latest_event = max(
    changelog["events"],
    key=lambda event: (event["date"], event["event_number"]),
)
latest_label_url = latest_event["label_url"]

print("Baseline label URL:", baseline_label_url)
print("Latest label date:", latest_event["date"])
print("Latest label URL:", latest_label_url)

## Generate the deterministic Section 1 diff

The existing changelog utilities normalize PDF wrapping into logical blocks. The diff includes replacements, insertions, and deletions plus one neighboring block of context on each side. No LLM is involved in this step.

In [ ]:
section_pair = load_section_pair_from_cache(
    SECTION_CACHE_JSON,
    baseline_label_url=baseline_label_url,
    latest_label_url=latest_label_url,
)
diff_hunks = build_section_diff_hunks(
    section_pair["baseline_section"],
    section_pair["latest_section"],
    context_blocks=1,
)
print(f"Deterministic diff hunks: {len(diff_hunks)}")

In [ ]:
for hunk in diff_hunks:
    print("=" * 100)
    print(hunk["hunk_id"], hunk["change_type"].upper())
    print("\nBASELINE CONTEXT BEFORE:", *hunk["baseline_context_before"], sep="\n")
    print("\nBASELINE CHANGED TEXT:", hunk["baseline_text"], sep="\n")
    print("\nLATEST CHANGED TEXT:", hunk["latest_text"], sep="\n")
    print("\nLATEST CONTEXT AFTER:", *hunk["latest_context_after"], sep="\n")

## Load the existing curated indications

In [ ]:
existing_indications = load_existing_indications(
    EXISTING_INDICATIONS_JSON, document_id="doc:fda.opdivo"
)
for indication in existing_indications:
    print(f"{indication['id']}: {indication['indication']}\n")

## Inspect the exact assessment prompt

In [ ]:
assessment_prompt = build_revision_identification_prompt(
    existing_indications, diff_hunks
)
print(assessment_prompt)

## Assess which existing indications were revised

This is the notebook's only LLM call. It must assess every existing indication and cite the deterministic hunk IDs supporting each revision.

In [ ]:
revision_assessment = identify_revised_indications(
    existing_indications, diff_hunks
)
print("Verified:", revision_assessment["verified"])
print("Verification errors:", revision_assessment["verification_errors"])
print("Status counts:", Counter(
    item["status"] for item in revision_assessment["assessments"]
))

## Review every assessment

In [ ]:
for assessment in revision_assessment["assessments"]:
    print("=" * 100)
    print(assessment["existing_indication_id"], assessment["status"].upper())
    print(assessment["existing_indication"]["indication"])
    print("Relevant hunks:", assessment["relevant_hunk_ids"])
    print("Changes:")
    for change in assessment["changes"]:
        print(" -", change)
    print("Reason:", assessment["reason"])

## Inspect the proposal prompt for one revised indication

The assessed `changes` define the proposal's scope. The cited hunk supplies current-label wording, but it may also contain neighboring indications that must not enter the proposal.

In [ ]:
revised_assessments = [
    item
    for item in revision_assessment["assessments"]
    if item["status"] == "revised"
]
if not revised_assessments:
    print("No revised indications require proposals.")
else:
    example_assessment = revised_assessments[0]
    proposal_prompt = build_label_diff_revision_prompt(
        example_assessment["existing_indication"],
        example_assessment,
        example_assessment["relevant_hunks"],
    )
    print(proposal_prompt)

## Propose minimal updates for revised indications

This makes one focused LLM call per revised indication. It may update only `indication`, `description`, and the three `raw_*` fields. Dates, URLs, IDs, and other provenance remain unchanged.

In [ ]:
revision_proposals = [
    propose_revised_indication(
        assessment["existing_indication"],
        assessment,
        diff_hunks,
    )
    for assessment in revised_assessments
]
print(f"Generated proposals: {len(revision_proposals)}")

In [ ]:
for proposal in revision_proposals:
    print("=" * 100)
    print("Existing indication ID:", proposal["existing_indication_id"])
    print("Supporting hunks:", proposal["supporting_hunk_ids"])
    if not proposal["changes"]:
        print("No target-specific field update was proposed.")
    for field, new_value in proposal["changes"].items():
        print(f"\nFIELD: {field}")
        print("EXISTING:", next(
            item["existing_indication"].get(field)
            for item in revised_assessments
            if item["existing_indication_id"] == proposal["existing_indication_id"]
        ))
        print("PROPOSED:", new_value)